# Lesson 07 - नियोजन डिझाइन नमुना

हा नोटबुक Microsoft Agent Framework वापरून AI एजंटसाठी **नियोजन डिझाइन नमुना** दर्शवितो.
आपण शिकाल की कसे गुंतागुंतीची प्रवास विनंत्या संरचित उपकार्यांमध्ये विभागायच्या,
त्यांना विशेष एजंट्सना नियुक्त करायचे,
आणि परिणामी आराखडा कसा अंमलात आणायचा — सगळं Pydantic मॉडेल्सच्या मदतीने संरचित आउटपुट वापरून.


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Task Decomposition

टास्क डिकंपोजीशन ही प्लॅनिंग डिझाइन पॅटर्नची मूळ बाब आहे. एका एजंटला संपूर्ण गुंतागुंतीचा विनंती हाताळण्याऐवजी, आपण समस्या लहान, चांगल्या प्रकारे व्याख्यित **उपटास्क** मध्ये विभागतो. प्रत्येक उपटास्क एका तज्ञ एजंटला (उदा. फ्लाइट्स, हॉटेल्स, क्रियाकलाप) स्पष्ट प्राधान्यक्रम आणि अवलंबित्व क्रमवारीसह दिला जातो.

हा दृष्टिकोन काही फायदे प्रदान करतो:
- **स्पष्टता**: प्रत्येक उपटास्कची एक जबाबदारी असते.
- **समानांतरतेने चालणे**: स्वतंत्र उपटास्क एकाच वेळी चालवता येतात.
- **विश्वसनीयता**: अपयश फक्त वेगवेगळ्या उपटास्कमध्ये मर्यादित असते.
- **बजेट ट्रॅकिंग**: खर्च प्रत्येक उपटास्कसाठी अंदाजित केला जातो आणि एकत्र सादर केला जातो.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## संरचित आउटपुटसह नियोजन एजंट तयार करणे

नियोजन एजंट हा एक **फ्रंट डेस्क समन्वयक** म्हणून कार्य करतो. उच्च-स्तरीय प्रवास विनंती दिल्यास तो एक संरचित `TravelPlan` तयार करतो — विनंतीचे उपकार्यात भागांमध्ये विघटन करणे, प्राधान्ये सेट करणे, आणि अवलंबित्व ओळखणे जेणेकरून कन्सियरज किंवा अंमलबजावणी स्तर काम पार पाडू शकेल.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## विशेषज्ञ साधनांसह योजना अंमलबजावणी

एकदा फ्रंट डेस्क एजंटने एक संरचित योजना तयार केली की, **कंसीयार्ज एजंट** ती अंमलात आणतो.
प्रत्येक विशेषज्ञ साधन एका उपकार्यातील वर्गासाठी (फ्लाइट्स, हॉटेल्स, क्रियाकलाप) जबाबदार असते. कंसीयार्ज
योजनेतील उपकार्ये अवलंबित्वाच्या क्रमाने पाहतो आणि प्रत्येक उपकार्य योग्य साधनाकडे पाठवतो.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## सारांश

या धड्यात तुम्ही AI एजंटसाठी **योजना डिझाइन पॅटर्न** शिकली:

1. **कार्य विभागणी** — एक फ्रंट डेस्क नियोजन एजंट गुंतागुंतीच्या प्रवास विनंतीला Pydantic मॉडेल्स वापरून संरचित उपकार्यांमध्ये विभागतो, प्रत्येक उपकार्यात प्राधान्ये आणि अवलंबित्वांसह तज्ञ एजंटला नियुक्त करतो.
2. **संरचित आउटपुट** — `response_format` देऊन एजंट स्वातंत्र्यचकित मजकूराऐवजी मान्यताप्राप्त `TravelPlan` ऑब्जेक्ट परत करतो, ज्यामुळे खालील प्रक्रिया विश्वासार्ह होते.
3. **योजना कार्यान्वयन** — एक कन्सिएर्ज एजंट उपकार्यांमधून पार पाडण्यासाठी तज्ञ साधने (`book_flight`, `reserve_hotel`, `book_activity`) वापरतो आणि परिणामांची नोंद करतो.

हा पॅटर्न *काय करायचे* (योजना) आणि *कसे करायचे* (कार्यान्वयन) वेगळे करतो, ज्यामुळे एजंट अधिक विभागीय, तपासण्यायोग्य आणि वाढवण्यास सुलभ होतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील आहोत, तरी कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेतील त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला जावा. महत्त्वपूर्ण माहितीसाठी व्यावसायिक मानवी अनुवाद करणे शिफारसीय आहे. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थलहरीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
